# Gold Layer Notebook

1. Read from silver_carbon_intensity_regional
2. Add a 7-day rolling average of carbon_intensity per region
3. Add lag features (t-1, t-2, t-48, t-336) per region
4. Interpolate nulls (Gold's job, not Silver's)
5. Write to gold_carbon_intensity_regional  

In [0]:
# Read from the silver layer delta table

df_gold = spark.read.table("silver_carbon_intensity_regional")
df_gold.show()

In [0]:
# Add 7 day rolling average to as feature

from pyspark.sql.window import Window
import pyspark.sql.functions as F

window = Window.partitionBy("region_id").orderBy("settlement_period").rowsBetween(-335, 0)

df_gold = df_gold.withColumn("rolling_avg_7day", F.avg("carbon_intensity").over(window))

df_gold.show()

In [0]:
# Quick sanity check of the rolling averages - choose a busier region

display(df_gold.filter(df_gold.region_name=="West Midlands").limit(20))

In [0]:
# Add lag features (t-1, t-2, t-48, t-336) per region


lag_window = Window.partitionBy("region_id").orderBy("settlement_period")

# t-1 lag (30 mins)

df_gold = df_gold.withColumn("lag_1", F.lag("carbon_intensity").over(lag_window))

# t-2 lag (1 hour)

df_gold = df_gold.withColumn("lag_2", F.lag("carbon_intensity", 2).over(lag_window))

# t-48 lag (24 hours)

df_gold = df_gold.withColumn("lag_48", F.lag("carbon_intensity", 48).over(lag_window))

# t-336 lag (336)

df_gold = df_gold.withColumn("lag_336", F.lag("carbon_intensity", 336).over(lag_window))

df_gold.show()

In [0]:
df_gold.filter(F.col("lag_336").isNotNull()).show(20)

In [0]:
# Skip interpolation for now because there are no Null values. 

# Write to the gold layer delta table
df_gold.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("gold_carbon_intensity_regional")

In [0]:
display(df_gold.describe())